In [1]:
import pandas as pd
import numpy as np
import json
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv("../dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("../dataset/tmdb_5000_credits.csv")

In [3]:
merged_df = movies.merge(credits, left_on="id", right_on="movie_id")

In [4]:
merged_df = merged_df[["genres", "keywords", "overview", "cast", "crew", "title_x", "movie_id"]]
merged_df = merged_df.rename(columns={"title_x": "title"})

In [5]:
merged_df = merged_df.dropna()

In [6]:
merged_df["genres"] = merged_df["genres"].apply(
    lambda x: [i["name"].replace(" ", "") for i in json.loads(x)]
)

In [7]:
merged_df["keywords"] = merged_df["keywords"].apply(
    lambda x: [i["name"].replace(" ", "") for i in json.loads(x)]
)

In [8]:
merged_df["cast"] = merged_df["cast"].apply(
    lambda x: [i["name"].replace(" ", "") for i in json.loads(x)][:3]
)

In [9]:
merged_df["crew"] = merged_df["crew"].apply(
    lambda x: [i["name"].replace(" ", "") for i in json.loads(x) if i["job"]=="Director"][:1]
)

In [10]:
merged_df["overview"] = merged_df["overview"].apply(lambda x: x.split())

In [11]:
merged_df["tags"] = (
    merged_df["overview"]
    + merged_df["genres"]
    + merged_df["keywords"]
    + merged_df["cast"]
    + merged_df["crew"]
)

In [ ]:
merged_df["tags"] = merged_df["tags"].apply(lambda x: " ".join(x).lower())

In [12]:
merged_df = merged_df[["movie_id", "title", "tags"]]

In [14]:
stop_words = set(stopwords.words("english"))
ps = PorterStemmer()
merged_df["tags"] = merged_df["tags"].apply(
    lambda x: [
        ps.stem(token) for token in re.findall(r"\b\w+(?:'\w+)?\b", x) if token not in stop_words
    ]
)

In [15]:
cv = CountVectorizer(tokenizer=lambda x: x, lowercase=False, max_features=5000)
vectors = cv.fit_transform(merged_df["tags"])

/home/vikas/Projects/Movie-Recommender-System/env/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [16]:
similarity = cosine_similarity(vectors)

In [17]:
def recommend(movie):
    movie_index = np.where(merged_df["title"]==movie)[0][0]
    distances = similarity[movie_index]
    movies_list = sorted(enumerate(distances), reverse=True, key=lambda x: x[1])[1:6]
    for index, score in movies_list:
        print(merged_df.iloc[index].title)

In [18]:
recommend("Batman Begins")

The Dark Knight
The Dark Knight Rises
Batman & Robin
Batman v Superman: Dawn of Justice
Batman


In [19]:
recommend("Avatar")

Falcon Rising
Aliens
Titan A.E.
Aliens vs Predator: Requiem
Ender's Game


In [20]:
recommend("Harry Potter and the Philosopher's Stone")

Harry Potter and the Chamber of Secrets
Harry Potter and the Half-Blood Prince
Harry Potter and the Goblet of Fire
Harry Potter and the Prisoner of Azkaban
Harry Potter and the Order of the Phoenix


In [21]:
recommend("Spider-Man")

Spider-Man 3
Spider-Man 2
The Amazing Spider-Man
The Amazing Spider-Man 2
Arachnophobia


In [22]:
recommend("Superman")

Superman II
Superman Returns
Superman III
Man of Steel
Superman IV: The Quest for Peace


In [23]:
recommend("The Avengers")

Avengers: Age of Ultron
Iron Man 3
Captain America: The Winter Soldier
Captain America: Civil War
Captain America: The First Avenger


In [24]:
recommend("Pirates of the Caribbean: At World's End")

Pirates of the Caribbean: Dead Man's Chest
Pirates of the Caribbean: The Curse of the Black Pearl
Pirates of the Caribbean: On Stranger Tides
VeggieTales: The Pirates Who Don't Do Anything
Life of Pi
